# Logistic Regression — Diabetes Risk Prediction
### Binary Classification | Healthcare Domain

---

## Problem Statement

A hospital's preventive care department wants to **screen patients for diabetes risk** during routine checkups, before expensive lab-confirmed diagnosis is ordered.

**Business/Clinical Goal:** Build a classification model that predicts whether a patient is **likely Diabetic (1)** or **Non-Diabetic (0)**, using easily measurable clinical indicators collected during a standard checkup (glucose level, BMI, blood pressure, age, family history, etc.) **without** needing an expensive confirmatory lab test upfront.

**Why this matters:**
- Diabetes often develops silently  early risk flagging allows **doctors to recommend lifestyle changes or further testing** before the disease progresses.
- A risk-screening model helps **prioritize high-risk patients** for confirmatory blood tests (like HbA1c), saving cost and time for low-risk patients.
- Understanding which factors drive risk up (e.g., glucose, BMI) helps **clinicians and patients understand actionable health levers**.

**Expected Output:**
- For each patient, the model outputs a **probability of being diabetic** (0 to 1) and a **binary classification** — Diabetic or Non-Diabetic — based on a 0.5 decision threshold.
- Example: *"Patient X has an 82% probability of being diabetic → flagged as Diabetic, recommend HbA1c test."*

---



## step 1 Data Understanding


##  Dataset Variables (Real-World Clinical Columns)

| Column | Type | Real-World Meaning | Risk Direction |
|---|---|---|---|
| `glucose_level` | Numerical | Plasma glucose concentration (mg/dL) from a glucose tolerance test | Higher → higher diabetes risk |
| `bmi` | Numerical | Body Mass Index (weight/height²) | Higher → higher risk (obesity is a major risk factor) |
| `blood_pressure` | Numerical | Diastolic blood pressure (mm Hg) | Higher → associated with higher risk |
| `age` | Numerical | Patient's age in years | Older → higher risk |
| `insulin_level` | Numerical | 2-hour serum insulin (mu U/ml) | Abnormal levels → higher risk |
| `skin_thickness` | Numerical | Triceps skinfold thickness (mm) — a proxy for body fat | Higher → higher risk |
| `family_history` | Categorical | Yes / No — close relative with diabetes | Yes → higher risk (genetic predisposition) |
| `physical_activity` | Categorical | Low / Moderate / High — weekly activity level | Lower activity → higher risk |
| `diabetic` | **Target** | 1 = Diabetic, 0 = Non-Diabetic | What we want to predict |

>Dataset is **clean** — no missing values, no outliers, realistic clinical value ranges (mirrors structure of well-known clinical diabetes datasets).

## Step 2 — Import Libraries

In [ ]:
#!pip install numpy pandas scikit-learn matplotlib seaborn

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import (
    accuracy_score, confusion_matrix,
    classification_report, roc_auc_score, roc_curve,
    ConfusionMatrixDisplay
)
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.facecolor': '#FAFAFA',
    'axes.facecolor':   '#F2F4F7',
    'axes.grid':        True,
    'grid.alpha':       0.4,
    'font.size':        12,
})
sns.set_palette("husl")
print("All libraries imported!")


## Step 3 — Exploratory Data Analysis (EDA)

### Why EDA for a clinical screening model?
- Confirms whether the target classes (Diabetic/Non-Diabetic) are balanced
- Shows whether clinical markers (glucose, BMI) genuinely differ between diabetic and non-diabetic patients — validating that the data makes medical sense
- Surfaces correlated features and possible multicollinearity before modelling

> In healthcare ML, EDA is also a **sanity check** — if glucose level doesn't separate diabetic from non-diabetic patients at all, something is wrong with the data or labels.


In [ ]:
df=pd.read_csv('diabetes_risk.csv')

In [ ]:
print("=" * 55)
print("DATASET INFO")
print("=" * 55)
df.info()


In [ ]:
print("=" * 55)
print("STATISTICAL SUMMARY")
print("=" * 55)
df.describe().round(2)


In [ ]:
print("=" * 55)
print("MISSING VALUES COUNT")
print("=" * 55)
missing     = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_df  = pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct})
missing_df  = missing_df[missing_df['Missing Count'] > 0]
print(missing_df)


In [ ]:
# ── Class Balance ──────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle('Target Class Distribution', fontsize=14, fontweight='bold')

labels = ['Non-Diabetic (0)', 'Diabetic (1)']
counts = df['diabetic'].value_counts().sort_index()
colors = ['#2ECC71', '#E74C3C']

axes[0].bar(labels, counts.values, color=colors, edgecolor='white', linewidth=1.5, width=0.5)
axes[0].set_title('Count of Each Class', fontweight='bold')
axes[0].set_ylabel('Number of Patients')
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 8, str(v), ha='center', fontsize=13, fontweight='bold')

axes[1].pie(counts.values, labels=labels, colors=colors, autopct='%1.1f%%',
            startangle=90, textprops={'fontsize': 12},
            wedgeprops={'edgecolor': 'white', 'linewidth': 2})
axes[1].set_title('Class Proportion', fontweight='bold')

plt.tight_layout()
plt.savefig('class_balance.png', dpi=120)
plt.show()
print("\nWHY check class balance? An imbalanced dataset (e.g. 95% non-diabetic)")
print("   could make a model lazily predict 'non-diabetic' for everyone and still")
print("   look 95% accurate — while missing every actual diabetic patient.")
print("   Our dataset is balanced — the model must genuinely learn the patterns.")


In [ ]:
# ── Numerical Feature Distributions by Class ──────────────────
num_cols = ['glucose_level', 'bmi', 'blood_pressure', 'age', 'insulin_level', 'skin_thickness']

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()
fig.suptitle('Clinical Feature Distributions by Diabetes Status',
             fontsize=15, fontweight='bold', y=1.01)

for i, col in enumerate(num_cols):
    diabetic_vals    = df[df['diabetic'] == 1][col]
    non_diabetic_vals = df[df['diabetic'] == 0][col]

    axes[i].hist(non_diabetic_vals, bins=30, alpha=0.6, color='#2ECC71',
                 label='Non-Diabetic (0)', edgecolor='white', linewidth=0.4)
    axes[i].hist(diabetic_vals, bins=30, alpha=0.6, color='#E74C3C',
                 label='Diabetic (1)', edgecolor='white', linewidth=0.4)
    axes[i].set_title(col, fontweight='bold')
    axes[i].set_ylabel('Count')
    axes[i].legend(fontsize=9)

    axes[i].axvline(diabetic_vals.mean(), color='#A93226', linewidth=2,
                    linestyle='--')
    axes[i].axvline(non_diabetic_vals.mean(), color='#1A8A4A', linewidth=2,
                    linestyle='--')

plt.tight_layout()
plt.savefig('numerical_distributions.png', dpi=120)
plt.show()
print("\n HOW TO READ: If diabetic (red) histogram sits CLEARLY to the right of")
print("   non-diabetic (green) → that feature is clinically meaningful for risk.")
print("   glucose_level and bmi should show the clearest separation —")
print("   matching real clinical knowledge that they are primary diabetes indicators.")


In [ ]:
# ── Box Plots by Class ─────────────────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()
fig.suptitle('Box Plots — Clinical Feature Spread by Diabetes Status',
             fontsize=15, fontweight='bold', y=1.01)

for i, col in enumerate(num_cols):
    data_to_plot = [
        df[df['diabetic'] == 0][col].values,
        df[df['diabetic'] == 1][col].values
    ]
    bp = axes[i].boxplot(data_to_plot, patch_artist=True,
                         labels=['Non-Diabetic (0)', 'Diabetic (1)'],
                         medianprops=dict(color='black', linewidth=2.5))
    bp['boxes'][0].set_facecolor('#2ECC71'); bp['boxes'][0].set_alpha(0.6)
    bp['boxes'][1].set_facecolor('#E74C3C'); bp['boxes'][1].set_alpha(0.6)
    axes[i].set_title(col, fontweight='bold')
    axes[i].set_ylabel('Value')

plt.tight_layout()
plt.savefig('boxplots_by_class.png', dpi=120)
plt.show()
print("\nHOW TO READ: Boxes at DIFFERENT heights → the feature meaningfully")
print("separates diabetic from non-diabetic patients. Same height → weak feature.")


In [ ]:
# ── Categorical Features vs Target ────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle('Categorical Risk Factors vs Diabetes Status', fontsize=14, fontweight='bold')

for ax, col in zip(axes, ['family_history', 'physical_activity']):
    ct = pd.crosstab(df[col], df['diabetic'], normalize='index') * 100
    ct.columns = ['Non-Diabetic %', 'Diabetic %']
    ct.plot(kind='bar', ax=ax, color=['#2ECC71', '#E74C3C'],
            edgecolor='white', linewidth=0.8, rot=15)
    ax.set_title(f'{col} vs Diabetes Status', fontweight='bold')
    ax.set_ylabel('Percentage (%)')
    ax.set_xlabel('')
    ax.legend(loc='upper right')
    for container in ax.containers:
        ax.bar_label(container, fmt='%.1f%%', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.savefig('categorical_vs_target.png', dpi=120)
plt.show()
print("\n HOW TO READ: Taller red bar = that category has a higher diabetes rate.")
print("   Patients with 'Yes' family history and 'Low' physical activity should")
print("   show higher diabetic percentages — matching known clinical risk factors.")


## Step 4 — Correlation Heatmap

### Why correlation for a clinical risk model?
- Confirms which **clinical markers correlate most strongly** with diabetic status — clinically, we'd expect `glucose_level` and `bmi` to lead
- Flags **multicollinearity** (e.g., `bmi` and `skin_thickness` may be related, since both relate to body fat)

> In Logistic Regression, correlated predictors can make individual coefficients unstable — useful to know before interpreting feature importance.


In [ ]:
df_enc = df.copy()
le = LabelEncoder()
df_enc['family_history']    = le.fit_transform(df_enc['family_history'])
df_enc['physical_activity'] = le.fit_transform(df_enc['physical_activity'])

corr = df_enc.corr()

fig, ax = plt.subplots(figsize=(11, 9))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdYlGn',
            center=0, vmin=-1, vmax=1, ax=ax,
            linewidths=0.6, linecolor='white',
            annot_kws={"size": 10, "weight": "bold"},
            cbar_kws={"shrink": 0.8})
ax.set_title('🌡️ Correlation Heatmap — Clinical Features', fontsize=14, fontweight='bold', pad=15)
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.savefig('correlation_heatmap.png', dpi=120)
plt.show()

print("\n Correlation of each feature with TARGET (diabetic):")
target_corr = corr['diabetic'].drop('diabetic').sort_values(ascending=False)
for feat, val in target_corr.items():
    bar = '█' * int(abs(val) * 30)
    sign = '+' if val >= 0 else '-'
    print(f"  {feat:<22} {sign}{abs(val):.3f}  {bar}")
print("\nglucose_level and bmi typically show the strongest correlation —")
print("consistent with their role as primary clinical diabetes risk indicators.")


## Step 5 — How Logistic Regression Works

### Why not Linear Regression for diabetes risk?
Linear Regression outputs **unbounded continuous values** (e.g. -0.5 or 2.3) — meaningless as a probability of disease.
We need an output **strictly between 0 and 1** that can be interpreted as a *risk probability*.

### The Sigmoid Function
$$\sigma(z) = \frac{1}{1 + e^{-z}}$$

where:

$$z = \beta_0 + \beta_1 x_1 + \beta_2 x_2 + \cdots + \beta_n x_n$$

### Decision Rule
$$\hat{y} = \begin{cases} 1 \text{ (Diabetic)} & \text{if } \sigma(z) \geq 0.5 \\ 0 \text{ (Non-Diabetic)} & \text{if } \sigma(z) < 0.5 \end{cases}$$

> In practice, a clinician might lower this threshold (e.g. to 0.3) to **catch more at-risk patients early**, even if it means more false alarms — this is a common real-world tradeoff in healthcare screening.


In [ ]:
# ── Visualise the Sigmoid Function ────────────────────────────
z      = np.linspace(-10, 10, 300)
sigmoid = 1 / (1 + np.exp(-z))

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(z, sigmoid, color='#3498DB', linewidth=3, label='σ(z) = 1 / (1 + e⁻ᶻ)')
ax.axhline(0.5, color='#E74C3C', linewidth=1.8, linestyle='--', label='Decision boundary (0.5)')
ax.axhline(1.0, color='gray', linewidth=1, linestyle=':', alpha=0.5)
ax.axhline(0.0, color='gray', linewidth=1, linestyle=':', alpha=0.5)
ax.axvline(0, color='black', linewidth=0.8, linestyle='-', alpha=0.3)

ax.fill_between(z, 0.5, sigmoid, where=(sigmoid >= 0.5),
                alpha=0.12, color='#E74C3C', label='Diabetic region (≥ 0.5)')
ax.fill_between(z, sigmoid, 0.5, where=(sigmoid < 0.5),
                alpha=0.12, color='#2ECC71', label='Non-Diabetic region (< 0.5)')

ax.set_xlabel('z  (linear combination of clinical features)', fontsize=12)
ax.set_ylabel('σ(z)  =  Probability of Diabetes', fontsize=12)
ax.set_title('Sigmoid Function — Converting Clinical Score into Risk Probability',
             fontsize=13, fontweight='bold')
ax.legend(fontsize=10)
ax.set_ylim(-0.05, 1.05)
plt.tight_layout()
plt.savefig('sigmoid_function.png', dpi=120)
plt.show()

print("KEY INSIGHT:")
print("   z → -∞  :  σ(z) → 0   (very low diabetes risk)")
print("   z =  0  :  σ(z) = 0.5 (borderline — exactly at decision threshold)")
print("   z → +∞  :  σ(z) → 1   (very high diabetes risk)")
print("\n   The model learns β values that push high-glucose, high-BMI patients")
print("   toward z >> 0, and healthy-marker patients toward z << 0.")


## Step 6 — Feature Engineering

### Steps:
1. **Label Encoding** — Convert `family_history` and `physical_activity` text to numbers
2. **Train-Test Split** — 80% train / 20% test, stratified to preserve class balance
3. **Feature Scaling** — Standardise numerical features (mean=0, std=1)

> No missing value handling or outlier removal needed — our dataset is already clean.


In [ ]:
df_model = df.copy()

print("=" * 50)
print("6A: LABEL ENCODING")
print("=" * 50)

le = LabelEncoder()
for col in ['family_history', 'physical_activity']:
    original = sorted(df_model[col].unique())
    df_model[col] = le.fit_transform(df_model[col])
    encoded  = sorted(df_model[col].unique())
    print(f"  '{col}': {original} → {encoded}")

print("\nLogistic Regression requires numerical inputs — text must be encoded first.")


In [ ]:
print("=" * 50)
print("6B: TRAIN / TEST SPLIT")
print("=" * 50)

X = df_model.drop(columns='diabetic')
y = df_model['diabetic']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"  Training samples : {X_train.shape[0]}")
print(f"  Testing  samples : {X_test.shape[0]}")
print(f"\n  Train class balance: {y_train.value_counts().to_dict()}")
print(f"  Test  class balance: {y_test.value_counts().to_dict()}")
print("\nstratify=y ensures both train and test sets have the same diabetic ratio.")


In [ ]:
print("=" * 50)
print("6C: STANDARD SCALING")
print("=" * 50)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

print(f"  glucose_level — raw mean   : {X_train['glucose_level'].mean():.2f}")
print(f"  glucose_level — scaled mean: {X_train_scaled[:, 0].mean():.6f}  (≈ 0)")
print(f"  glucose_level — scaled std : {X_train_scaled[:, 0].std():.6f}   (≈ 1)")
print("\nAll features scaled. Ready for modelling!")


## Step 7 — Train Logistic Regression Model

The model learns the coefficients (β values) that **maximise the likelihood** of correctly classifying patients as Diabetic or Non-Diabetic, using **Maximum Likelihood Estimation (MLE)**.


In [ ]:
model = LogisticRegression(random_state=42, max_iter=1000)
model.fit(X_train_scaled, y_train)

y_pred       = model.predict(X_test_scaled)
y_pred_proba = model.predict_proba(X_test_scaled)[:, 1]   # probability of Diabetic

feature_names = X.columns.tolist()

print("Logistic Regression model trained!")
print(f"\n  Intercept (β₀) : {model.intercept_[0]:.4f}")
print(f"\n  Learned Coefficients:")
print(f"  {'Feature':<20} {'Coefficient':>12}  {'Effect'}")
print("  " + "-"*52)
for feat, coef in zip(feature_names, model.coef_[0]):
    effect = "↑ Diabetes risk" if coef > 0 else "↓ Diabetes risk"
    print(f"  {feat:<20} {coef:>12.4f}  ({effect})")

print("\nLarger positive β → feature strongly increases diabetes risk")
print("Larger negative β → feature is protective (lowers risk)")


## Step 8 — Manual Risk Calculation for One Fixed Patient

Let's manually trace, using plain Python, exactly how the model calculates a diabetes risk probability for one patient.


In [ ]:
# ── Fixed patient values ────────────────────────────────────────
patient = {
    'glucose_level':     165,
    'bmi':               32.5,
    'blood_pressure':    88,
    'age':               45,
    'insulin_level':     180,
    'skin_thickness':    30,
    'family_history':    1,      # 1 = Yes (after label encoding)
    'physical_activity': 0       # 0 = High, 1 = Low, 2 = Moderate (alphabetical encoding)
}

print("=" * 55)
print("Patient — Fixed Input Values")
print("=" * 55)
for k, v in patient.items():
    print(f"  {k:<20} : {v}")

input_df     = pd.DataFrame([patient])
input_scaled = scaler.transform(input_df)

intercept    = model.intercept_[0]
coefficients = model.coef_[0]

print("\n" + "=" * 60)
print("Step-by-Step Calculation")
print("   z = β₀ + β₁x₁ + β₂x₂ + ... + βₙxₙ")
print("=" * 60)
print(f"\n  {'Term':<22} {'β':>10}  {'x (scaled)':>12}  {'β × x':>12}")
print("  " + "-"*60)

z = intercept
print(f"  {'Intercept (β₀)':<22} {intercept:>10.4f}  {'—':>12}  {intercept:>12.4f}")
for feat, coef, x_val in zip(feature_names, coefficients, input_scaled[0]):
    contrib = coef * x_val
    z      += contrib
    print(f"  {feat:<22} {coef:>10.4f}  {x_val:>12.4f}  {contrib:>12.4f}")

print("  " + "-"*60)
print(f"  {'z  (linear score)':<22} {'':>10}  {'':>12}  {z:>12.4f}")

probability = 1 / (1 + np.exp(-z))
decision    = "🔴 DIABETIC"  if probability >= 0.5 else "🟢 NON-DIABETIC"

print(f"\n  Sigmoid:  σ(z) = 1 / (1 + e^(-{z:.4f}))")
print(f"           σ(z) = {probability:.6f}")
print(f"\n  Decision threshold : 0.5")
print(f"  Predicted probability of Diabetes : {probability:.4f} ({probability*100:.2f}%)")
print(f"\n  ══════════════════════════════════")
print(f"   Final Decision : {decision}")
print(f"  ══════════════════════════════════")

model_output = model.predict(input_scaled)[0]
model_proba  = model.predict_proba(input_scaled)[0][1]
print(f"\n model.predict()       : {model_output}  ({'Diabetic' if model_output==1 else 'Non-Diabetic'})")
print(f" model.predict_proba() : {model_proba:.6f}")
print(f" Manual vs sklearn match  : {abs(probability - model_proba) < 1e-6}")


##  Step 9 — Model Evaluation

### Metrics for Clinical Classification:

| Metric | Formula | Clinical Meaning |
|---|---|---|
| **Accuracy** | Correct / Total | % of all patients correctly classified |
| **Precision** | TP / (TP + FP) | Of all patients flagged Diabetic, how many really are? |
| **Recall (Sensitivity)** | TP / (TP + FN) | Of all actual diabetic patients, how many did we catch? |
| **F1-Score** | 2 × (P × R) / (P + R) | Balance between Precision and Recall |
| **ROC-AUC** | Area under ROC curve | Overall ability to distinguish diabetic from non-diabetic |
<br>
>**Clinically, Recall is critical here** — missing a diabetic patient (False Negative) is far more costly than a false alarm, since an undiagnosed diabetic patient may develop complications.


In [ ]:
acc = accuracy_score(y_test, y_pred)
auc = roc_auc_score(y_test, y_pred_proba)

print("=" * 55)
print("MODEL EVALUATION RESULTS")
print("=" * 55)
print(f"  Accuracy  : {acc:.4f}  ({acc*100:.2f}%)")
print(f"  ROC-AUC   : {auc:.4f}")

print("\n" + "=" * 55)
print("CLASSIFICATION REPORT")
print("=" * 55)
print(classification_report(y_test, y_pred,
      target_names=['Non-Diabetic (0)', 'Diabetic (1)']))

print("HOW TO READ:")
print("  Precision = When model flags 'Diabetic', how often is it correct?")
print("  Recall    = Of all actual diabetic patients, how many did the model catch?")
print("  F1-Score  = Single score balancing both Precision and Recall")
print("  Support   = Actual number of patients in each class")


## Step 10 — Model Evaluation Visualizations

In [ ]:
# ── Confusion Matrix ──────────────────────────────────────────
cm = confusion_matrix(y_test, y_pred)
tn, fp, fn, tp = cm.ravel()

fig, axes = plt.subplots(1, 2, figsize=(15, 6))
fig.suptitle('🔍 Confusion Matrix Analysis', fontsize=14, fontweight='bold')

disp = ConfusionMatrixDisplay(confusion_matrix=cm,
                               display_labels=['Non-Diabetic (0)', 'Diabetic (1)'])
disp.plot(ax=axes[0], colorbar=False, cmap='Blues')
axes[0].set_title('Confusion Matrix (Counts)', fontweight='bold')

cm_pct = confusion_matrix(y_test, y_pred, normalize='true') * 100
disp2  = ConfusionMatrixDisplay(confusion_matrix=cm_pct.round(1),
                                 display_labels=['Non-Diabetic (0)', 'Diabetic (1)'])
disp2.plot(ax=axes[1], colorbar=False, cmap='Reds')
axes[1].set_title('Confusion Matrix (% per True Class)', fontweight='bold')

plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=120)
plt.show()

print("=" * 55)
print("Confusion Matrix Breakdown:")
print("=" * 55)
print(f"  True  Negative (TN) = {tn:4d}  → Correctly predicted Non-Diabetic")
print(f"  False Positive (FP) = {fp:4d}  → Predicted Diabetic but was Non-Diabetic")
print(f"  False Negative (FN) = {fn:4d}  → Predicted Non-Diabetic but was Diabetic MISSED CASE")
print(f"  True  Positive (TP) = {tp:4d}  → Correctly predicted Diabetic")
print(f"\n  Total test patients   = {tn+fp+fn+tp}")
print(f"  Correctly classified  = {tn+tp}  ({(tn+tp)/(tn+fp+fn+tp)*100:.1f}%)")
print(f"\n CLINICAL NOTE: False Negatives (FN={fn}) are the most concerning —")
print(f" these are diabetic patients the model failed to flag for follow-up testing.")


In [ ]:
# ── ROC Curve ─────────────────────────────────────────────────
fpr, tpr, thresholds = roc_curve(y_test, y_pred_proba)

fig, ax = plt.subplots(figsize=(8, 7))
ax.plot(fpr, tpr, color='#E74C3C', linewidth=3,
        label=f'Logistic Regression (AUC = {auc:.3f})')
ax.plot([0, 1], [0, 1], color='gray', linewidth=1.5,
        linestyle='--', label='Random Classifier (AUC = 0.500)')
ax.fill_between(fpr, tpr, alpha=0.08, color='#E74C3C')

ax.set_xlabel('False Positive Rate (FPR)  →  1 - Specificity', fontsize=12)
ax.set_ylabel('True Positive Rate (TPR)  →  Recall / Sensitivity', fontsize=12)
ax.set_title('ROC Curve — Diabetes Risk Model Discrimination Ability',
             fontsize=13, fontweight='bold')
ax.legend(fontsize=11, loc='lower right')
ax.set_xlim([0, 1]); ax.set_ylim([0, 1.02])

plt.tight_layout()
plt.savefig('roc_curve.png', dpi=120)
plt.show()

print("HOW TO READ the ROC Curve:")
print("   X-axis (FPR) : How often we wrongly flag a healthy patient as diabetic")
print("   Y-axis (TPR) : How often we correctly catch an actual diabetic patient")
print("   Dashed line  : Random guessing — no better than a coin flip")
print(f"   AUC = {auc:.3f}  → Model is {auc*100:.1f}% better than random guessing at separating classes")
print("   AUC = 1.0    : Perfect model | AUC = 0.5 : Useless model")


In [ ]:
# ── Predicted Probability Distribution ───────────────────────
fig, ax = plt.subplots(figsize=(11, 5))

proba_diabetic     = y_pred_proba[y_test == 1]
proba_non_diabetic = y_pred_proba[y_test == 0]

ax.hist(proba_non_diabetic, bins=30, alpha=0.65, color='#2ECC71',
        label='Actual: Non-Diabetic (0)', edgecolor='white', linewidth=0.4)
ax.hist(proba_diabetic, bins=30, alpha=0.65, color='#E74C3C',
        label='Actual: Diabetic (1)', edgecolor='white', linewidth=0.4)
ax.axvline(0.5, color='black', linewidth=2, linestyle='--', label='Decision threshold (0.5)')

ax.set_xlabel('Predicted Probability of Diabetes', fontsize=12)
ax.set_ylabel('Count', fontsize=12)
ax.set_title('Predicted Risk Probability Distribution by Actual Status',
             fontsize=13, fontweight='bold')
ax.legend(fontsize=11)

plt.tight_layout()
plt.savefig('probability_distribution.png', dpi=120)
plt.show()

print("HOW TO READ:")
print("   Red bars (Diabetic) should pile up near 1.0 — model is confident")
print("   Green bars (Non-Diabetic) should pile up near 0.0 — model is confident")
print("   Overlap near 0.5 = borderline patients the model is uncertain about —")
print("   clinically, these are good candidates for a confirmatory lab test.")


In [ ]:
# ── Feature Importance (Coefficients) ────────────────────────
fig, ax = plt.subplots(figsize=(10, 6))

coef_df = pd.DataFrame({
    'Feature'     : feature_names,
    'Coefficient' : model.coef_[0]
}).sort_values('Coefficient')

colors_bar = ['#2ECC71' if c < 0 else '#E74C3C' for c in coef_df['Coefficient']]
bars = ax.barh(coef_df['Feature'], coef_df['Coefficient'],
               color=colors_bar, edgecolor='white', linewidth=0.8)
ax.axvline(0, color='black', linewidth=1.2)

for bar, val in zip(bars, coef_df['Coefficient']):
    x_pos = val + 0.01 if val >= 0 else val - 0.01
    ha    = 'left' if val >= 0 else 'right'
    ax.text(x_pos, bar.get_y() + bar.get_height()/2,
            f'{val:.3f}', va='center', ha=ha, fontsize=10, fontweight='bold')

ax.set_title('Feature Coefficients  (Red = increases diabetes risk  |  Green = protective)',
             fontsize=12, fontweight='bold')
ax.set_xlabel('Coefficient Value (on scaled features)', fontsize=12)
plt.tight_layout()
plt.savefig('feature_coefficients.png', dpi=120)
plt.show()

print("HOW TO READ:")
print("   The coefficient tells us the change in LOG-ODDS of being diabetic")
print("   per 1 standard deviation increase in that feature (scaled).")
print("   Longer red bar  → strongest risk-increasing clinical factor")
print("   Longer green bar → strongest protective factor (e.g. high physical activity)")
print("\n   Clinically, glucose_level and bmi should top the list —")
print("   matching established medical knowledge on diabetes risk factors.")


## Step 11 — Diabetes Risk Prediction Function for New Patients

In [ ]:
def predict_diabetes_risk(glucose_level, bmi, blood_pressure, age,
                          insulin_level, skin_thickness,
                          family_history, physical_activity):
    """
    Predict diabetes risk for a new patient.

    Parameters:
    -----------
    glucose_level      : int    — Plasma glucose level in mg/dL (70–200)
    bmi                : float  — Body Mass Index
    blood_pressure     : int    — Diastolic blood pressure in mm Hg (60–120)
    age                : int    — Patient age (21–80)
    insulin_level      : int    — 2-hour serum insulin in mu U/ml (15–300)
    skin_thickness     : int    — Triceps skinfold thickness in mm (7–50)
    family_history     : str    — 'Yes' or 'No'
    physical_activity  : str    — 'Low', 'Moderate', or 'High'

    Returns: predicted label (0 or 1) and diabetes risk probability
    """

    family_map   = {'No': 0, 'Yes': 1}
    activity_map = {'High': 0, 'Low': 1, 'Moderate': 2}

    if family_history not in family_map:
        raise ValueError("family_history must be 'Yes' or 'No'")
    if physical_activity not in activity_map:
        raise ValueError(f"physical_activity must be one of {list(activity_map.keys())}")

    input_df = pd.DataFrame([{
        'glucose_level':     glucose_level,
        'bmi':               bmi,
        'blood_pressure':    blood_pressure,
        'age':               age,
        'insulin_level':     insulin_level,
        'skin_thickness':    skin_thickness,
        'family_history':    family_map[family_history],
        'physical_activity': activity_map[physical_activity]
    }])

    input_scaled = scaler.transform(input_df)
    prediction    = model.predict(input_scaled)[0]
    probability   = model.predict_proba(input_scaled)[0][1]
    decision      = "🔴 DIABETIC"     if prediction == 1 else "🟢 NON-DIABETIC"
    confidence    = probability if prediction == 1 else 1 - probability

    print("=" * 50)
    print(" Diabetes Risk Prediction")
    print("=" * 50)
    print(f"  Glucose Level      : {glucose_level} mg/dL")
    print(f"  BMI                : {bmi}")
    print(f"  Blood Pressure     : {blood_pressure} mm Hg")
    print(f"  Age                : {age}")
    print(f"  Insulin Level      : {insulin_level} mu U/ml")
    print(f"  Skin Thickness     : {skin_thickness} mm")
    print(f"  Family History     : {family_history}")
    print(f"  Physical Activity  : {physical_activity}")
    print("-" * 50)
    print(f"  Diabetes Risk Probability : {probability:.4f} ({probability*100:.2f}%)")
    print(f"  Model Confidence          : {confidence*100:.2f}%")
    print(f"  Decision                  : {decision}")
    if prediction == 1:
        print(f"  Recommendation          : Refer for confirmatory HbA1c / fasting glucose test")
    else:
        print(f"  Recommendation          : Routine follow-up at next checkup")
    print("=" * 50)

    return prediction, probability


# ── Example Usage ──────────────────────────────────────────────
predict_diabetes_risk(
    glucose_level     = 165,
    bmi               = 32.5,
    blood_pressure    = 88,
    age               = 45,
    insulin_level     = 180,
    skin_thickness    = 30,
    family_history    = 'Yes',
    physical_activity = 'Low'
)


In [ ]:
# Compare: same patient profile but healthy lifestyle markers
predict_diabetes_risk(
    glucose_level     = 95,
    bmi               = 23.0,
    blood_pressure    = 75,
    age               = 45,
    insulin_level     = 80,
    skin_thickness    = 18,
    family_history    = 'No',
    physical_activity = 'High'
)
print("\n Notice how much LOWER the risk is with healthy glucose, BMI, and activity —")
print(" this matches clinical understanding that lifestyle factors meaningfully")
print(" affect diabetes risk, even though some risk (age, genetics) is fixed.")


## Step 12 — Summary & Key Takeaways

### Problem Recap
We built a classification model to predict **diabetes risk** based on routinely measurable clinical markers (glucose, BMI, blood pressure, insulin, age) and lifestyle/genetic factors (family history, physical activity) — supporting **early risk screening** before expensive lab-confirmed diagnosis.

### End-to-End Pipeline:

| Step | Action | Key Insight |
|---|---|---|
| 1 | Problem statement | Diabetes risk screening to prioritize confirmatory testing |
| 2 | Dataset creation | 1000 patients, 8 clinical features, balanced 50/50 classes |
| 3 | EDA — class balance | Balanced classes → fair model learning |
| 4 | EDA — distributions | Glucose & BMI show clear separation between classes |
| 5 | Correlation heatmap | Confirms glucose_level & bmi as top predictors |
| 6 | Sigmoid function | Converts linear risk score z into a probability (0–1) |
| 7 | Feature engineering | Encode → Stratified Split → Scale |
| 8 | Model training | MLE finds β values maximising correct classification |
| 9 | Manual calculation | Trace z → sigmoid → decision for one patient |
| 10 | Evaluation metrics | Accuracy, Precision, Recall, F1, ROC-AUC |
| 11 | Visualisations | Confusion matrix, ROC curve, probability distribution, coefficients |
| 12 | Prediction function | Ready-to-use risk screening tool for new patients |

### Clinical Takeaway
The model confirms what clinicians already know:
- **Glucose level** and **BMI** are the strongest diabetes risk indicators
- **Family history** adds meaningful genetic risk
- **High physical activity** is protective, lowering risk

> **Important note:** This is a screening/triage tool, not a diagnostic replacement. A "Diabetic" flag means *"recommend confirmatory lab testing"* — not a clinical diagnosis. Final diagnosis always requires a qualified medical professional and lab-confirmed tests (e.g., HbA1c, fasting glucose).


In [ ]:
print("=" * 55)
print("FINAL MODEL PERFORMANCE SUMMARY")
print("=" * 55)
print(f"  Dataset         : {len(df)} patients · 8 clinical features")
print(f"  Train / Test    : {len(X_train)} / {len(X_test)} patients")
print(f"  Class Balance   : {y_test.value_counts().to_dict()}")
print()
print(f"  ┌─────────────────────────────────────────┐")
print(f"  │  Accuracy   :  {acc:.4f}  ({acc*100:.2f}%)         |")
print(f"  │  ROC-AUC    :  {auc:.4f}                   │")
print(f"  │  True  Positives (TP) : {tp:<5}           │")
print(f"  │  True  Negatives (TN) : {tn:<5}           │")
print(f"  │  False Positives (FP) : {fp:<5}           │")
print(f"  │  False Negatives (FN) : {fn:<5}           |")
print(f"  └─────────────────────────────────────────┘")
print()
print("   You now understand how Logistic Regression can power a")
print("   real-world clinical risk-screening tool — from raw patient")
print("   data → sigmoid math → evaluation → actionable recommendation.")
